<a href="https://colab.research.google.com/github/josemanuelmontanoortiz978-droid/PROYECTOS-ED1/blob/main/Ejercicios_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from __future__ import annotations
from abc import ABC, abstractmethod
from typing import Any, Optional
import ast
import inspect
import re


class Nodo:
    """Representa un nodo individual para estructuras enlazadas."""

    def __init__(self, dato: Any, siguiente: Optional[Nodo] = None) -> None:
        """Inicializa un nodo con su valor y referencia al siguiente nodo."""
        self.dato: Any = dato
        self.siguiente: Optional[Nodo] = siguiente


class EstructuraLineal(ABC):
    """Interfaz abstracta que define el contrato de una estructura lineal."""

    @abstractmethod
    def insertar(self, elemento: Any) -> None:
        """Inserta un nuevo elemento en la estructura."""
        pass

    @abstractmethod
    def eliminar(self) -> Any:
        """Elimina y retorna el elemento correspondiente según la política."""
        pass

    @abstractmethod
    def esta_vacia(self) -> bool:
        """Indica si la estructura no contiene elementos."""
        pass

    @abstractmethod
    def __len__(self) -> int:
        """Retorna la cantidad actual de elementos almacenados."""
        pass


class Pila(EstructuraLineal):
    """Implementación concreta de una Pila (LIFO) usando nodos enlazados."""

    def __init__(self) -> None:
        """Inicializa una pila vacía."""
        self._cima: Optional[Nodo] = None
        self._tamanio: int = 0

    def insertar(self, elemento: Any) -> None:
        """Apila un elemento en la cima en tiempo O(1)."""
        nuevo_nodo: Nodo = Nodo(elemento, self._cima)
        self._cima = nuevo_nodo
        self._tamanio += 1

    def eliminar(self) -> Any:
        """Desapila y retorna el elemento de la cima en tiempo O(1)."""
        if self.esta_vacia():
            raise IndexError("Error de subdesbordamiento: la pila está vacía.")
        assert self._cima is not None
        dato_extraido: Any = self._cima.dato
        self._cima = self._cima.siguiente
        self._tamanio -= 1
        return dato_extraido

    def esta_vacia(self) -> bool:
        """Verifica si la cima apunta a None."""
        return self._cima is None

    def __len__(self) -> int:
        """Devuelve el número de elementos en la pila."""
        return self._tamanio


class Cola(EstructuraLineal):
    """Implementación concreta de una Cola (FIFO) usando nodos enlazados."""

    def __init__(self) -> None:
        """Inicializa una cola vacía con punteros al frente y al final."""
        self._frente: Optional[Nodo] = None
        self._final: Optional[Nodo] = None
        self._tamanio: int = 0

    def insertar(self, elemento: Any) -> None:
        """Encola un elemento al final de la estructura en tiempo O(1)."""
        nuevo_nodo: Nodo = Nodo(elemento)
        if self.esta_vacia():
            self._frente = nuevo_nodo
            self._final = nuevo_nodo
        else:
            assert self._final is not None
            self._final.siguiente = nuevo_nodo
            self._final = nuevo_nodo
        self._tamanio += 1

    def eliminar(self) -> Any:
        """Desencola y retorna el elemento del frente en tiempo O(1)."""
        if self.esta_vacia():
            raise IndexError("Error de subdesbordamiento: la cola está vacía.")
        assert self._frente is not None
        dato_extraido: Any = self._frente.dato
        self._frente = self._frente.siguiente
        if self._frente is None:
            self._final = None
        self._tamanio -= 1
        return dato_extraido

    def esta_vacia(self) -> bool:
        """Verifica si el frente de la cola es None."""
        return self._frente is None

    def __len__(self) -> int:
        """Devuelve el número de elementos en la cola."""
        return self._tamanio


def procesar_estructura(estructura: EstructuraLineal, lotes: list[Any]) -> list[Any]:
    """Consume cualquier EstructuraLineal de forma polimórfica.

    Inserta todos los elementos de 'lotes' y luego los extrae uno a uno
    hasta vaciar la estructura, evidenciando el orden LIFO o FIFO.
    """
    for item in lotes:
        estructura.insertar(item)

    print(f"Clase recibida: {type(estructura).__name__} | Tamaño cargado: {len(estructura)}")

    orden_salida: list[Any] = []
    while not estructura.esta_vacia():
        orden_salida.append(estructura.eliminar())

    return orden_salida


# --- PRUEBA DE POLIMORFISMO ---
datos_entrada: list[str] = ["Tarea_1", "Tarea_2", "Tarea_3"]

mi_pila: EstructuraLineal = Pila()
mi_cola: EstructuraLineal = Cola()

print("Salida Pila (LIFO):", procesar_estructura(mi_pila, datos_entrada))
print("Salida Cola (FIFO):", procesar_estructura(mi_cola, datos_entrada))


Clase recibida: Pila | Tamaño cargado: 3
Salida Pila (LIFO): ['Tarea_3', 'Tarea_2', 'Tarea_1']
Clase recibida: Cola | Tamaño cargado: 3
Salida Cola (FIFO): ['Tarea_1', 'Tarea_2', 'Tarea_3']


In [ ]:
def auditar_codigo_fuente(codigo_fuente: str) -> list[str]:
    """Analiza un bloque de código Python mediante AST en busca de infracciones.

    Revisa:
    1. Docstrings faltantes en clases y funciones/métodos.
    2. Nombres no válidos (PascalCase para clases, snake_case para funciones).
    3. Funciones o métodos públicos sin type hints en parámetros o retorno.
    """
    arbol: ast.AST = ast.parse(codigo_fuente)
    errores: list[str] = []

    patron_clase: re.Pattern[str] = re.compile(r"^[A-Z][a-zA-Z0-9]*$")
    patron_funcion: re.Pattern[str] = re.compile(r"^(_?[a-z][a-z0-9_]*|__[a-z][a-z0-9_]*__)$")

    for nodo in ast.walk(arbol):
        # 1. Validación de Clases
        if isinstance(nodo, ast.ClassDef):
            if ast.get_docstring(nodo) is None:
                errores.append(f"[Línea {nodo.lineno}] Clase '{nodo.name}' sin docstring.")
            if not patron_clase.match(nodo.name):
                errores.append(
                    f"[Línea {nodo.lineno}] Nombre de clase inválido '{nodo.name}' (debe ser PascalCase)."
                )

        # 2. Validación de Funciones y Métodos
        elif isinstance(nodo, (ast.FunctionDef, ast.AsyncFunctionDef)):
            if ast.get_docstring(nodo) is None:
                errores.append(f"[Línea {nodo.lineno}] Función/método '{nodo.name}' sin docstring.")

            if not patron_funcion.match(nodo.name):
                errores.append(
                    f"[Línea {nodo.lineno}] Nombre de función inválido '{nodo.name}' (debe ser snake_case)."
                )

            # Solo exigir type hints en funciones/métodos públicos (no inician con '_')
            es_publica: bool = not nodo.name.startswith("_")
            if es_publica:
                if nodo.returns is None:
                    errores.append(
                        f"[Línea {nodo.lineno}] Función pública '{nodo.name}' sin type hint de retorno."
                    )

                argumentos: list[ast.arg] = (
                    nodo.args.posonlyargs + nodo.args.args + nodo.args.kwonlyargs
                )
                for arg in argumentos:
                    if arg.arg in ("self", "cls"):
                        continue
                    if arg.annotation is None:
                        errores.append(
                            f"[Línea {nodo.lineno}] Parámetro '{arg.arg}' en '{nodo.name}' sin type hint."
                        )

    return errores


# --- EJECUCIÓN DEL AUDITOR ---
# 1. Auditamos nuestras clases y funciones reales de la celda anterior
objetos_a_auditar: list[Any] = [Nodo, EstructuraLineal, Pila, Cola, procesar_estructura]
codigo_propio: str = "\n\n".join(inspect.getsource(obj) for obj in objetos_a_auditar)

reporte_propio: list[str] = auditar_codigo_fuente(codigo_propio)
print("=== AUDITORÍA DE NUESTRA IMPLEMENTACIÓN ===")
if not reporte_propio:
    print("0 errores encontrados. El código cumple al 100% con los estándares.")
else:
    for err in reporte_propio:
        print(" -", err)

# 2. Prueba negativa con un código defectuoso para validar el detector
codigo_con_fallos: str = """
class mala_clase:
    def MetodoPublicoSinTipos(self, x, y: int):
        return x + y
"""

print("\n=== PRUEBA CON CÓDIGO DEFECTUOSO ===")
for err in auditar_codigo_fuente(codigo_con_fallos):
    print(" -", err)